
## Highlighting the Contextual Issue

Let's first ask the model about a topic that it hasn't seen during pretraining, specifically, let's ask it who is teaching the currrent class.



In [ ]:
!pip install langchain_huggingface

In [ ]:
!pip install -U langchain langchain-mistralai

In [ ]:
!pip install langchain_classic
!pip install -U langchain-community

In [ ]:
!pip install chromadb

In [ ]:
from langchain_mistralai import ChatMistralAI
import os
from google.colab import userdata

llm = ChatMistralAI(model="mistral-tiny",  mistral_api_key=userdata.get('MISTRAL_AI_KEY')) # Or other Mistral models

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI assistant."),
        ("user", "{input}")
    ])

chain = prompt | llm | StrOutputParser()

In [ ]:
response = chain.invoke({"input": "Who teaches the AI Safety and Security course, Fall 2026 course at Katz?"})
print(response)

As expected, Mistral hasn't the slightest idea who's teaching the current class. We need to create a basic RAG system that will retrieve the right context to answer that question.

First, let's grab the syllabus from the local directory and do some basic chunking.

In [ ]:

from langchain_classic.document_loaders import TextLoader, DirectoryLoader

from pathlib import Path
docs_dir = "."
loader = DirectoryLoader(
    docs_dir, glob=f"ai_safety_and_secure_syllabus.txt", show_progress=True, loader_cls=TextLoader

)
data = loader.load()

In [ ]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from tqdm import tqdm

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    length_function=len,

)
text_docs = text_splitter.split_documents(data)

Let's look at a few chunks

In [ ]:

def display_doc_split(docs):

    """display information about the split documents"""

    print("\n--- Document Chunks Information ---")
    print(f"Number of document chunks: {len(docs)}")
    print(f"Sample chunk:\n{docs[0].page_content}\n")


display_doc_split(text_docs)

Next, we need a way to embed and index the chunks. We'll use a `sentence-transformers` embedding model for this. The database we'll be using is `chromadb`

In [ ]:

from langchain_huggingface import HuggingFaceEmbeddings
# create embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
from langchain_classic.vectorstores import Chroma

persist_directory = "db"
vectordb = Chroma.from_documents(
    documents=text_docs, embedding=embeddings, persist_directory=persist_directory

)


In [ ]:
from langchain.tools import tool
@tool(response_format="content_and_artifact")
def retrieve_context(query: str, k=10):
    """Retrieve information to help answer a query."""
    retriever = vectordb.as_retriever(

    search_type="similarity",

    search_kwargs={"k":k},
    )
    relevant_docs = retriever.invoke(query)
    return relevant_docs
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in relevant_docs
    )
    return serialized, relevant_docs


In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
format_docs(text_docs)

Now let's create a new prompt that forces the LLM to refer to the results of querying the RAG database given the prompt.

In [ ]:
# imports
from langchain_core.prompts import PromptTemplate

# Create the Prompt Template
prompt_template = """Use the context provided to answer
the user's question below. If you do not know the answer
based on the context provided, tell the user that you do
not know the answer to their question based on the context
provided and that you are sorry.

Also provide a summary of the document that you used to answer
the user's question.

context: {context}

question: {query}

answer: """

# Create Prompt Instance from template
custom_rag_prompt = PromptTemplate.from_template(prompt_template)

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Define retriever (this was missing)
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10},
)

# Create the RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | custom_rag_prompt
    | llm
    | StrOutputParser()
)

# Query the RAG Chain
rag_chain.invoke(
  "Who is the instructor for the AI Safety and Security course?"
)

But our RAG system is fairly brittle: the system is unable to answer any questions that doesn't involve the provided context

In [ ]:
rag_chain.invoke(
  "When was America discovered?"
)

To solve this, we need to use the Agent paradigm. Specifically, we need a Planning component that allows the Agent to access its base weights when asked general questions but should access the RAG database when asked questions related to the class.